In [0]:
# ============================================================
# GOLD PRODUCT SALES
# Grain: 1 row = 1 product
# Incremental + MERGE + Watermark
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from delta.tables import DeltaTable


# ============================================================
# 2. CONFIGURATION
# ============================================================

PIPELINE_NAME = "gold_product_sales"


# ============================================================
# 3. PATHS
# ============================================================

orders_silver_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "silver/orders/"
)

products_silver_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "silver/products/"
)

gold_product_sales_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "gold/product_sales/"
)

control_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)


print("================================================")
print("GOLD PRODUCT SALES PIPELINE")
print("================================================")


# ============================================================
# 4. READ SILVER ORDERS
# ============================================================

orders_df = (
    spark.read
    .format("delta")
    .load(orders_silver_path)
)

print(
    "Silver order records:",
    orders_df.count()
)


# ============================================================
# 5. READ SILVER PRODUCTS
# ============================================================

products_df = (
    spark.read
    .format("delta")
    .load(products_silver_path)
)

print(
    "Silver product records:",
    products_df.count()
)


# ============================================================
# 6. READ CONTROL TABLE
# ============================================================

control_table = DeltaTable.forPath(
    spark,
    control_path
)

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)


# ============================================================
# 7. CHECK WHETHER GOLD PRODUCT WATERMARK EXISTS
# ============================================================

watermark_exists = (
    control_df
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .count() > 0
)

print(
    "gold_product_sales watermark exists:",
    watermark_exists
)


# ============================================================
# 8. CREATE WATERMARK ROW IF MISSING
# ============================================================

watermark_schema = StructType([
    StructField(
        "pipeline_name",
        StringType(),
        False
    ),
    StructField(
        "last_processed_timestamp",
        TimestampType(),
        True
    ),
    StructField(
        "last_run_id",
        StringType(),
        True
    ),
    StructField(
        "updated_at",
        TimestampType(),
        True
    )
])


if not watermark_exists:

    initial_watermark_df = spark.createDataFrame(
        [
            (
                PIPELINE_NAME,
                None,
                None,
                None
            )
        ],
        schema=watermark_schema
    )

    (
        initial_watermark_df
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )

    print(
        "Created gold_product_sales watermark."
    )


# ============================================================
# 9. READ CURRENT GOLD PRODUCT WATERMARK
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)

gold_product_last_processed_timestamp = (
    control_df
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .select(
        "last_processed_timestamp"
    )
    .collect()[0][0]
)

print(
    "Current Gold Product watermark:",
    gold_product_last_processed_timestamp
)


# ============================================================
# 10. IDENTIFY INCREMENTAL ORDERS
# ============================================================

if gold_product_last_processed_timestamp is None:

    print("First Gold Product Sales run.")
    print("Processing all Silver orders.")

    incremental_orders_df = orders_df

else:

    print(
        "Processing orders after:",
        gold_product_last_processed_timestamp
    )

    incremental_orders_df = (
        orders_df
        .filter(
            F.col("ingestion_timestamp")
            > gold_product_last_processed_timestamp
        )
    )


incremental_count = incremental_orders_df.count()


print(
    "Incremental order records:",
    incremental_count
)


# ============================================================
# 11. NO NEW DATA
# ============================================================

if incremental_count == 0:

    print("==============================================")
    print("No new orders found.")
    print("Gold Product Sales is already up to date.")
    print("Watermark will NOT be updated.")
    print("==============================================")

    dbutils.notebook.exit(
        "SUCCESS: No new records for Gold Product Sales."
    )


# ============================================================
# 12. FIND AFFECTED PRODUCTS
# ============================================================

affected_products_df = (
    incremental_orders_df
    .select(
        "product_id"
    )
    .distinct()
)


print(
    "Affected products:",
    affected_products_df.count()
)

display(
    affected_products_df
    .orderBy("product_id")
    .limit(20)
)


# ============================================================
# 13. READ ALL SILVER ORDERS FOR AFFECTED PRODUCTS
# ============================================================
#
# IMPORTANT:
#
# We don't calculate Gold using only the incremental orders.
#
# If Product 200001 already had 10,000 orders and today's
# batch adds 500 orders, Gold must recalculate Product 200001
# using all 10,500 Silver orders.
#
# ============================================================

affected_orders_df = (
    orders_df
    .join(
        affected_products_df,
        on="product_id",
        how="inner"
    )
)


print(
    "Orders for affected products:",
    affected_orders_df.count()
)


# ============================================================
# 14. JOIN ORDERS WITH PRODUCTS
# ============================================================

affected_orders_products_df = (
    affected_orders_df.alias("o")
    .join(
        products_df.alias("p"),
        F.col("o.product_id")
        ==
        F.col("p.product_id"),
        "left"
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.product_id").alias("product_id"),
        F.col("o.quantity").alias("quantity"),
        F.col("o.sales_amount").alias("sales_amount"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.category").alias("category"),
        F.col("p.brand").alias("brand")
    )
)


print(
    "Joined order-product records:",
    affected_orders_products_df.count()
)


# ============================================================
# 15. CALCULATE PRODUCT SALES METRICS
# ============================================================

affected_product_sales_df = (
    affected_orders_products_df
    .groupBy(
        "product_id",
        "product_name",
        "category",
        "brand"
    )
    .agg(
        F.sum(
            "sales_amount"
        ).alias("total_sales"),

        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("total_quantity"),

        F.round(
            F.avg("sales_amount"),
            2
        ).alias("average_order_value")
    )
)


print(
    "Product Gold records to process:"
)

display(
    affected_product_sales_df
    .orderBy("product_id")
    .limit(20)
)


# ============================================================
# 16. VALIDATE SOURCE GRAIN
# ============================================================
#
# Gold Product grain:
#
# 1 row = 1 product
#
# Therefore product_id must be unique in the source
# before MERGE.
#
# ============================================================

duplicate_product_source_df = (
    affected_product_sales_df
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_product_source_count = (
    duplicate_product_source_df.count()
)


print(
    "Duplicate product IDs in Gold source:",
    duplicate_product_source_count
)


if duplicate_product_source_count > 0:

    display(
        duplicate_product_source_df
        .orderBy("product_id")
    )

    raise Exception(
        "Gold Product Sales validation failed: "
        "multiple source rows exist for the same product_id."
    )


# ============================================================
# 17. CREATE OR MERGE GOLD PRODUCT TABLE
# ============================================================

gold_product_exists = DeltaTable.isDeltaTable(
    spark,
    gold_product_sales_path
)


if not gold_product_exists:

    print(
        "Gold Product Sales table does not exist."
    )

    print(
        "Creating Gold Product Sales table."
    )

    (
        affected_product_sales_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(gold_product_sales_path)
    )

    print(
        "Gold Product Sales table created."
    )


else:

    print(
        "Gold Product Sales table exists."
    )

    gold_product_table = DeltaTable.forPath(
        spark,
        gold_product_sales_path
    )

    (
        gold_product_table.alias("target")
        .merge(
            affected_product_sales_df.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        "Gold Product Sales MERGE completed."
    )


# ============================================================
# 18. READ GOLD TABLE
# ============================================================

gold_product_df = (
    spark.read
    .format("delta")
    .load(gold_product_sales_path)
)


print(
    "Gold Product Sales record count:",
    gold_product_df.count()
)


display(
    gold_product_df
    .orderBy("product_id")
    .limit(20)
)


# ============================================================
# 19. VALIDATE GOLD PRODUCT GRAIN
# ============================================================

duplicate_gold_products_df = (
    gold_product_df
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_gold_product_count = (
    duplicate_gold_products_df.count()
)


print(
    "Duplicate product IDs in Gold:",
    duplicate_gold_product_count
)


if duplicate_gold_product_count > 0:

    display(
        duplicate_gold_products_df
        .orderBy("product_id")
    )

    raise Exception(
        "Gold Product Sales validation failed: "
        "duplicate product_id values exist."
    )


# ============================================================
# 20. CALCULATE PROCESSED TIMESTAMP
# ============================================================
#
# IMPORTANT:
#
# Use MAX(ingestion_timestamp) from the incremental
# Silver records that were actually processed.
#
# DO NOT use current_timestamp() as the watermark.
#
# ============================================================

gold_product_processed_timestamp = (
    incremental_orders_df
    .agg(
        F.max(
            "ingestion_timestamp"
        ).alias(
            "max_ingestion_timestamp"
        )
    )
    .collect()[0][
        "max_ingestion_timestamp"
    ]
)


print(
    "Gold Product processed timestamp:",
    gold_product_processed_timestamp
)


# ============================================================
# 21. CREATE RUN ID
# ============================================================

gold_product_run_id = (
    f"GOLD_PRODUCT_SALES_"
    f"{gold_product_processed_timestamp.strftime('%Y%m%d%H%M%S')}"
)


print(
    "Gold Product run ID:",
    gold_product_run_id
)


# ============================================================
# 22. CREATE WATERMARK DATAFRAME
# ============================================================

gold_product_watermark_df = spark.createDataFrame(
    [
        (
            PIPELINE_NAME,
            gold_product_processed_timestamp,
            gold_product_run_id,
            None
        )
    ],
    schema=watermark_schema
).withColumn(
    "updated_at",
    F.current_timestamp()
)


print(
    "Gold Product watermark:"
)

display(
    gold_product_watermark_df
)


# ============================================================
# 23. UPDATE SHARED WATERMARK
# ============================================================

(
    control_table.alias("target")
    .merge(
        gold_product_watermark_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


print(
    "Gold Product watermark updated successfully."
)


# ============================================================
# 24. FINAL VERIFICATION
# ============================================================

final_gold_product_df = (
    spark.read
    .format("delta")
    .load(gold_product_sales_path)
)


final_product_watermark_df = (
    spark.read
    .format("delta")
    .load(control_path)
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
)


print("================================================")
print("GOLD PRODUCT SALES PIPELINE COMPLETE")
print("================================================")

print(
    "Silver order records:",
    orders_df.count()
)

print(
    "Silver product records:",
    products_df.count()
)

print(
    "Incremental order records:",
    incremental_count
)

print(
    "Affected products:",
    affected_products_df.count()
)

print(
    "Gold Product records:",
    final_gold_product_df.count()
)

print(
    "Duplicate product IDs:",
    duplicate_gold_product_count
)

print(
    "Final Gold Product watermark:"
)

display(
    final_product_watermark_df
)

print(
    "Final Gold Product Sales:"
)

display(
    final_gold_product_df
    .orderBy("product_id")
    .limit(20)
)

GOLD PRODUCT SALES PIPELINE
Silver order records: 55968
Silver product records: 50000
gold_product_sales watermark exists: True
Current Gold Product watermark: None
First Gold Product Sales run.
Processing all Silver orders.
Incremental order records: 55968
Affected products: 41990


product_id
200001
200002
200003
200004
200005
200006
200007
200008
200009
200010


Orders for affected products: 55968
Joined order-product records: 55968
Product Gold records to process:


product_id,product_name,category,brand,total_sales,total_orders,total_quantity,average_order_value
200001,Product 1,Nutrition,Nutriliife,281.36,3,9,56.27
200002,Product 2,Vitamins,Healthpro,490.56000000000006,3,16,98.11
200003,Product 3,Minerals,Nutriliife,25.48,1,2,25.48
200004,Product 4,Supplements,Healthpro,939.1499999999999,3,25,187.83
200005,Product 5,Nutrition,Nutriliife,186.39999999999998,2,4,46.6
200006,Product 6,Vitamins,Healthpro,415.01,3,9,83.0
200007,Product 7,Minerals,Nutriliife,708.16,3,16,141.63
200008,Product 8,Supplements,Healthpro,39.18,1,2,39.18
200009,Product 9,Nutrition,Nutriliife,1290.3999999999999,3,25,258.08
200010,Product 10,Vitamins,Healthpro,249.8,2,4,62.45


Duplicate product IDs in Gold source: 0
Gold Product Sales table exists.
Gold Product Sales MERGE completed.
Gold Product Sales record count: 41990


product_id,product_name,category,brand,total_sales,total_orders,total_quantity,average_order_value
200001,Product 1,Nutrition,Nutriliife,281.36,3,9,56.27
200002,Product 2,Vitamins,Healthpro,490.56000000000006,3,16,98.11
200003,Product 3,Minerals,Nutriliife,25.48,1,2,25.48
200004,Product 4,Supplements,Healthpro,939.1499999999999,3,25,187.83
200005,Product 5,Nutrition,Nutriliife,186.39999999999998,2,4,46.6
200006,Product 6,Vitamins,Healthpro,415.01,3,9,83.0
200007,Product 7,Minerals,Nutriliife,708.16,3,16,141.63
200008,Product 8,Supplements,Healthpro,39.18,1,2,39.18
200009,Product 9,Nutrition,Nutriliife,1290.3999999999999,3,25,258.08
200010,Product 10,Vitamins,Healthpro,249.8,2,4,62.45


... (2 outputs left out)

Gold Product watermark updated successfully.
GOLD PRODUCT SALES PIPELINE COMPLETE
Silver order records: 55968
Silver product records: 50000
Incremental order records: 55968
Affected products: 41990
Gold Product records: 41990
Duplicate product IDs: 0
Final Gold Product watermark:


pipeline_name,last_processed_timestamp,last_run_id,updated_at
gold_product_sales,2026-10-06T13:04:22.940Z,GOLD_PRODUCT_SALES_20261006130422,2026-10-06T15:01:30.672Z


Final Gold Product Sales:


product_id,product_name,category,brand,total_sales,total_orders,total_quantity,average_order_value
200001,Product 1,Nutrition,Nutriliife,281.36,3,9,56.27
200002,Product 2,Vitamins,Healthpro,490.56000000000006,3,16,98.11
200003,Product 3,Minerals,Nutriliife,25.48,1,2,25.48
200004,Product 4,Supplements,Healthpro,939.1499999999999,3,25,187.83
200005,Product 5,Nutrition,Nutriliife,186.39999999999998,2,4,46.6
200006,Product 6,Vitamins,Healthpro,415.01,3,9,83.0
200007,Product 7,Minerals,Nutriliife,708.16,3,16,141.63
200008,Product 8,Supplements,Healthpro,39.18,1,2,39.18
200009,Product 9,Nutrition,Nutriliife,1290.3999999999999,3,25,258.08
200010,Product 10,Vitamins,Healthpro,249.8,2,4,62.45
